# ⚡ `JEV-Set-150M` (`149,936,905` Params) — 100% Self-Contained "Run All" Colab Pro Runner
### Target Benchmark: [Jev Decision Index `v0.4.1`](https://huggingface.co/spaces/multimodalart/jev-decision-index) (`apolinario/decision-index` @ `25c6ec7b`)

**Autonomous Execution:** This notebook is **100% self-contained**.
You can upload **ONLY this notebook** to Google Colab, select GPU (A100 or L4), and click **Runtime → Run all** (`Ctrl+F9`).
It embeds the entire repository payload, automatically restores your workspace, links Google Drive for cross-session resumption, runs pre-flight verifications, executes the measured pilot ablation, and launches production training.

### Configuration
Set whether you want the fast pilot gate first (~5-10 min) followed by full training, or pilot only.

In [ ]:
# ⚙️ PIPELINE MODE SELECTION
# Options:
#   "pilot" : Runs real-weights verification, zero-synthetic real data build, teacher logits, measured Exp A/B ablation, and 1 short epoch (~5-10 mins).
#   "full"  : Runs pilot gate first to confirm hardware/data, then automatically proceeds into full Stage 1, Stage 2 distillation, Stage 3 equilibrium, 3-seed Model Souping, and official evaluation.
RUN_MODE = "full"  # <--- "full" runs everything autonomously from A to Z when you click 'Run all'!
print(f"✅ Configured RUN_MODE: {RUN_MODE}")

### 🌐 Optional: Connect Antigravity Terminal Directly to this Colab Runtime
If you want the AI assistant (Antigravity) to take terminal control and run commands directly inside this Colab machine:
1. Run the code cell below.
2. It will display a link `https://github.com/login/oauth/device` and an 8-character code.
3. Click the link, enter the code, and tell Antigravity in the chat: *"Tunnel is ready: [tunnel name]"*.
4. Antigravity will instantly connect its terminal and run everything for you!

*Note: If you prefer to just let Colab run automatically on its own, you do NOT need this tunnel cell — simply click **Runtime -> Run all**!*

In [ ]:
# 🔌 Start VS Code Remote Tunnel to connect Antigravity
!curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' | tar -xz
!./code tunnel --accept-server-license-terms


## Step 1: Detect GPU & Setup Mixed Precision

In [ ]:
import os
import sys
import torch

print("=== Hardware Detection ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"✅ Active CUDA GPU: {gpu_name} ({vram_gb} GB VRAM)")
    if "A100" in gpu_name:
        print("🚀 NVIDIA A100 detected: Full BF16 throughput active.")
    elif "L4" in gpu_name:
        print("✅ NVIDIA L4 detected: BF16 + gradient checkpointing active.")
    else:
        print(f"ℹ️ GPU {gpu_name} active. Precision will be configured automatically.")
else:
    print("⚠️ No CUDA GPU detected. For fast GPU training, go to: Runtime -> Change runtime type -> GPU (A100 or L4).")

## Step 2: Autonomous Self-Extraction & Google Drive Linking
If the repository isn't already present on disk, this cell automatically unpacks the embedded codebase and symlinks Google Drive (`/content/drive/MyDrive/jev-set-150m`) for automatic cross-session persistence.

In [ ]:
import base64
import io
import os
import shutil
import zipfile
from pathlib import Path

WORKSPACE_DIR = Path("/content/jev-set-150m")
is_colab = "google.colab" in sys.modules

# 1. Mount Google Drive for persistent checkpoint and dataset storage
if is_colab:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_root = Path("/content/drive/MyDrive/jev-set-150m")
        drive_root.mkdir(parents=True, exist_ok=True)
        (drive_root / "checkpoints").mkdir(exist_ok=True)
        (drive_root / "data/processed").mkdir(parents=True, exist_ok=True)
        print(f"✅ Google Drive persistent root linked: {drive_root}")
    except Exception as e:
        print(f"Drive notice (non-fatal): {e}")

# 2. Extract repository code autonomously if not already present
if not (WORKSPACE_DIR / "src/jev_set/modeling_jev_set.py").exists():
    print("Unpacking self-contained codebase payload...")
    payload = """..."""
    zip_bytes = base64.b64decode(payload)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall(str(WORKSPACE_DIR))
    print(f"✅ Self-extracted repository into {WORKSPACE_DIR}")

# 3. Symlink checkpoints & data to Google Drive for cross-session resumption
os.chdir(str(WORKSPACE_DIR))
if is_colab and Path("/content/drive/MyDrive/jev-set-150m").exists():
    drive_root = Path("/content/drive/MyDrive/jev-set-150m")
    for sub in ["checkpoints", "data/processed"]:
        local_sub = WORKSPACE_DIR / sub
        drive_sub = drive_root / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        if not local_sub.is_symlink():
            if local_sub.exists():
                for item in local_sub.iterdir():
                    target = drive_sub / item.name
                    if not target.exists():
                        if item.is_dir():
                            shutil.copytree(str(item), str(target))
                        else:
                            shutil.copy2(str(item), str(target))
                shutil.rmtree(str(local_sub))
            local_sub.symlink_to(drive_sub)
            print(f"✅ Linked {sub} -> {drive_sub}")

print(f"✅ Active working directory: {os.getcwd()}")

## Step 3: Install Pinned Dependencies & Decision Index Evaluation Kit

In [ ]:
!pip install -q \
  "torch>=2.5.1" \
  "transformers>=4.48.0" \
  "datasets>=3.2.0" \
  "accelerate>=1.2.1" \
  "safetensors>=0.5.0" \
  "scipy>=1.14.1" \
  "scikit-learn>=1.6.0" \
  "datasketch>=1.6.5" \
  "pyyaml>=6.0.2" \
  "tqdm>=4.67.1" \
  "pytest>=8.0.0"

# Verify / install apolinario/decision-index official runner
!if [ ! -d external/decision-index ]; then \
    git clone https://github.com/apolinario/decision-index.git external/decision-index && \
    cd external/decision-index && git checkout 25c6ec7b6b5f44a216c63c3a700de3a9c6d6699d && \
    pip install -q -e . && cd ../..; \
  else \
    cd external/decision-index && pip install -q -e . && cd ../..; \
  fi

print("✅ Dependencies and official kit installed.")

## Step 4: Strict Parameter Accounting & Real Weights Verification
Validates `149,936,905 <= 150,000,000` parameter ceiling, permutation equivariance (`< 1e-5`), 0 missing backbone keys against real `cross-encoder/ettin-reranker-150m-v1`, and 46 unit & edge-case tests.

In [ ]:
!python scripts/verify_params.py
!python scripts/verify_real_weights.py
!pytest tests/ -q

## Step 5: Fast Pilot Execution (Zero-Synthetic Data, Teacher Logits, Measured Exp A/B)
Executes pre-flight pilot gates with real data to confirm all hardware and dataset pipelines before full training.

In [ ]:
!bash scripts/run_colab_pro.sh pilot

## Step 6: Full Production Pipeline (Stage 1 → Stage 2 → Stage 3 → Model Soup → Official Evaluation)
Automatically proceeds when `RUN_MODE == "full"`. Checkpoints and logits persist directly to Google Drive.

In [ ]:
if RUN_MODE == "full":
    print("🚀 Starting Full Production Training Pipeline...")
    !bash scripts/run_colab_pro.sh full
else:
    print("ℹ️ RUN_MODE is set to 'pilot'. To run full multi-epoch training, set RUN_MODE = 'full' in Step 1 and re-run.")

## Step 7: Final Scorecard & Decontamination Audit Display

In [ ]:
import json
from pathlib import Path

print("=== JEV-Set-150M Final Summary ===")

score_file = Path("runs/jev-set-150m-v04-scorecard.json")
if score_file.exists():
    scorecard = json.loads(score_file.read_text())
    print("🏆 Official Decision Index Scorecard:")
    print(json.dumps(scorecard, indent=2))

audit_file = Path("data/processed/decontamination_audit.json")
if audit_file.exists():
    audit = json.loads(audit_file.read_text())
    print(f"\n✅ Decontamination Audit SHA-256: {audit.get('audit_signature_sha256')}")
    print(f"Total items decontaminated: {audit.get('candidates_inspected', 0)}")

data_summary = Path("data/processed/data_build_summary.json")
if data_summary.exists():
    s = json.loads(data_summary.read_text())
    print(f"\n✅ Real rows total: {s.get('real_rows_total')} | Synthetic rows: {s.get('synthetic_rows_total')}")
    print(f"Groups with real data: {s.get('groups_with_real_data')} / {s.get('dataset_groups_curated')}")